# Tools (도구)

OpenAI 플랫폼에서 사용할 수 있는 도구에 대한 개요는 다음과 같습니다. 
- 함수 호출
- 웹검색
- 원격 MCP 서버
- 파일 검색
- 이미지 생성
- 컴퓨터 사용

### 내장 도구 비용

* 참고 : [Built-in tools Pricing](https://openai.com/api/pricing/)

In [1]:
from dotenv import load_dotenv
load_dotenv() # read local .env file

True

In [5]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-5-nano"

### 웹검색 (Web Search)
모델이 응답을 생성하기 전에 최신 정보를 웹에서 검색할 수 있도록 허용

In [7]:
response = client.responses.create(
    model=Model,
    tools=[{"type": "web_search"}],
    input="오늘 한국의 종합주가지수는 얼마입니까?"
)

print(response.output_text)

현재 한국의 종합주가지수(KOSPI, KS11) 실시간 정보입니다.

- 지수: 5,516.55 포인트
- 당일 변동: +264.68 포인트 (+5.04%)
- 실시간 시각: 11:51:00 (한국시간)
- 전일 종가: 5,251.87 포인트

참고: 지수는 거래 중에 수시로 변동합니다. 필요하시면 오늘 장 마감 시점의 종가도 확인해 드리겠습니다. 출처: Investing.com의 코스피지수(KS11) 페이지. ([kr.investing.com](https://kr.investing.com/indices/kospi))


- 사용자 위치 지정

In [8]:
response = client.responses.create(
    model=Model,
    tools=[{
        "type": "web_search",
        "user_location": {
            "type": "approximate",
            "country": "KR",
            "city": "서울"
        }
    }],
    input="가로수 길에서 제일 핫한 레스토랑이 어디인가요?",
)

print(response.output_text)

가로수길(신사동)은 트렌드가 빨리 바뀌는 곳이라 “제일 핫한” 레스토랑은 시기나 취향에 따라 달라져요. 원하시는 방향을 알려주시면 최신 정보를 바탕으로 맞춤 추천해 드리겠습니다. 간단히 몇 가지를 확인하고 싶어요.

- 예산 범위가 어떻게 되나요? (예: 2인당 5만원대, 10만원대 이상 등)
- 분위기 선호가 있나요? (캐주얼/다이닝/룸살롱 느낌 등)
- 요리 종류 선호가 있나요? (한식/일식/양식/퓨전/카페 바 등)
- 인원과 예약 여부, 시간대(런치/디너)도 알려주실래요?
- 특별히 피하고 싶은 재료나 알레르기 정보가 있나요?

확인해 주시면 지금 바로 최신 정보를 검색해 4–6곳 정도의 핫플레이스 목록과 간단한 설명, 이용 팁(예약 팁 포함)까지 정리해 드리겠습니다. 검색해 드릴까요?


In [10]:
# "Financial Statements"라는 벡터 스토어 생성
vector_store = client.vector_stores.create(name="Financial Statements")
 
# OpenAI에 업로드할 파일 준비
file_paths = ["재무제표/goog-10k.pdf", "재무제표/brka-10k.pdf"]
file_streams = [open(path, "rb") for path in file_paths]
 
# 업로드 및 폴링 SDK 도우미를 사용하여 파일을 업로드하고 벡터 스토어에 추가,
# 파일 배치의 완료 상태를 폴링합니다.
file_batch = client.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)
 
# 이 작업의 결과를 보기 위해 상태 및 파일 개수를 출력할 수 있습니다.
print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2)


In [11]:
response = client.responses.create(
    model=Model,
    input="2023년 10월 말에 AAPL의 발행 주식 수는 얼마였나요?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }]
)
print(response)

Response(id='resp_0c0a2d9f5dcae9990069af8efd1ddc81978eb2e9319ac6477a', created_at=1773113085.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-nano-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0c0a2d9f5dcae9990069af8efd9d288197a7f65fcb6bd3c8b9', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFileSearchToolCall(id='fs_0c0a2d9f5dcae9990069af8f07cc4c8197932ae127d6961afd', queries=['2023년 10월 말에 AAPL의 발행 주식 수는 얼마였나요?', "What was Apple's shares outstanding at the end of October 2023?", 'AAPL shares outstanding October 2023 end of month', 'Apple October 2023 issued shares', 'Apple 2023년 10월 말 발행 주식 수'], status='completed', type='file_search_call', results=None), ResponseReasoningItem(id='rs_0c0a2d9f5dcae9990069af8f09e90c8197b1f47c5890832b1b', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0c0a2d9f5dcae9990069af8f16b6008197b3b41

In [12]:
response.output_text

'다음과 같이 진행하겠습니다.\n\n- 요청하신 AAPL(Apple Inc.)의 2023년 10월 말 발행 주식 수 데이터는 현재 업로드된 자료에서 찾을 수 없었습니다. 제가 확인한 파일들에는 Apple 관련 문서가 포함되어 있지 않고, 대신 Berkshire Hathaway나 Alphabet 등 타 기업의 10-K 같은 문서가 있습니다. 예를 들어 Berkshire Hathaway의 10-K가 포함되어 있습니다(예: brka-10k.pdf) , Alphabet(Goog) 10-K도 포함되어 있습니다(goog-10k.pdf) .\n\n- 추가로 확인된 파일들 정보 예시: brka-10k.pdf, goog-10k.pdf 등. 이들 문서의 내용은 Apple 데이터가 아닙니다  .\n\n질문에 정확히 답하려면 다음 중 하나를 선택해 주세요.\n- A. 발행 주식 수(issued shares)와 유통 주식 수(outstanding shares) 중 어느 것을 원하시는지 확인해 주세요. 일반적으로 "발행 주식 수"는 회사가 발행한 총 주식 수를 의미하고, "유통 주식 수"는 시장에 유통 중인 주식 수를 의미합니다.\n- B. 2023년 10월 말의 정확한 수치를 웹에서 확인해도 되면, Apple의 10-K/10-Q 혹은 공시 자료를 기반으로 수치를 찾아 드리겠습니다. 이 경우 자료 업로드를 요청드리거나 해당 수치를 직접 확인해 드릴 수 있도록 자료를 제공해 주세요.\n\n참고로, 제가 웹 검색을 통해 외부 자료를 가져오려면 별도 허용이 필요합니다. 현재 업로드된 파일만으로는 Apple의 수치를 확정해 드릴 수 없다는 점 양해 부탁드립니다.'

In [13]:
response = client.responses.create(
    model=Model,
    input="2023년 Google의 당기 순이익은 얼마였나요? BERKSHIRE HATHAWAY INC. 와 어느 쪽의 당기 순이익이 더 많았나요?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }]
)

print(response.output_text)

질문하신 2023년 수치는 현재 업로드된 파일들에서 확인할 수 없습니다. 제가 확인한 자료에선 Alphabet(구글)의 연도 수치가 2014–2016년대에 한정되어 있고(Berkshire Hathaway의 수치는 2018–2020년대 자료가 포함된 상태) 있습니다. 예를 들면 Alphabet의 2016년 순이익은 약 195억 달러로 기재되어 있습니다. 이처럼 2023년 수치는 파일에 포함되어 있지 않습니다. 

또한 Berkshire Hathaway의 2020년 순이익은 주주지분 귀속 순이익으로 약 425억 달러 수준으로 기재되어 있습니다. 2020년 수치가 해당 연도 기준으로 최신인 셈이고, 2023년 수치 역시 파일에 없습니다. 

정확한 2023년 수치를 원하시면, 웹에서 Alphabet의 2023년 연차보고서(또는 10-K)와 Berkshire Hathaway의 2023년 연차보고서를 찾아 비교해 드리겠습니다. 원하시면 제가 인터넷에서 공식 수치를 찾아 와서 두 회사의 2023년 당기 순이익을 제시하고, 어느 쪽이 더 큰지도 함께 알려드리겠습니다.


### Remote MCP
- 모델이 원격 MCP 서버를 사용하여 작업을 수행할 수 있도록 허용
- 모델 컨텍스트 프로토콜 (MCP)은 애플리케이션이 LLM에 도구와 컨텍스트를 제공하는 방식을 표준화
- Responses API의 MCP 도구를 통해 개발자는 모델에 원격 MCP 서버 에 호스팅된 도구에 대한 액세스 권한을 부여
- 원격 MCP 서버는 개발자와 조직이 인터넷을 통해 관리하며, Responses API와 같은 도구를 MCP 클라이언트에 제공하는 MCP 서버

In [6]:
#  DeepWiki MCP 서버를 사용하여 거의 모든 공개 GitHub 저장소에 대해 질문
resp = client.responses.create(
    model=Model,
    tools=[
        {
            "type": "mcp",
            "server_label": "deepwiki",
            "server_url": "https://mcp.deepwiki.com/mcp",
            "require_approval": "never",
        },
    ],
    # input="2025-03-26 버전의 MCP 사양에서는 어떤 전송 프로토콜이 지원됩니까?",
    input="ironmanciti/Generative_AI_4Days 에는 어떤 코드들이 있습니까?",
)

print(resp.output_text)

요약해 드리자면, ironmanciti/Generative_AI_4Days 저장소는 4일짜리 Generative AI 강좌용 코드/노트북 모음이며, 코드 파일은 주로 Jupyter Notebook에 담겨 있습니다. 또한 Google Workspace 연동용 자바스크립트 파일도 포함되어 있습니다.

주요 코드 구성 요약

- Day 1: Foundations (핵심 개념/실습 노트북)
  - 010_embedding.ipynb — 텍스트 임베딩 생성 및 사용 흐름
  - 020_Tokenizer_tiktoken.ipynb — tiktoken 기반 토크나이제이션
  - 150_autoregressive_language_generation.ipynb — 자동 회귀 언어 생성 예제

- Day 2: API Integration (OpenAI API 연동)
  - 200_Text_Generation-Prompts.ipynb — 프롬프트 엔지니어링 및 기본 텍스트 생성
  - 210_Structured_Output.ipynb — 구조화된 출력(Pydantic 등) 활용
  - 220_Function_Call.ipynb — 함수 호출 패턴
  - 230_Conversation_state.ipynb — 대화 상태 관리
  - 235_Reasoning.ipynb — 고급 추론 모델 활용
  - 250_Streaming.ipynb — 스트리밍 응답 처리

- Day 3: Advanced Features (고급 기능)
  - 260_File_inputs.ipynb — PDF 등 파일 입력 처리
  - 270_Tools.ipynb — 호스팅 도구/툴 체계
  - 300_Interacting_CLIP.ipynb — CLIP 기반 멀티모달 처리
  - 310_Image_Generation.ipynb — 이미지 생성
  - 320_Images_and_vision.ipynb — 컴퓨터 비전/이미지 처리

- Day 4: AI Agent Systems / Practical Apps
  - 

In [14]:
tool_list = []

for item in resp.output:
    if item.type == "mcp_list_tools":
        for tool in item.tools:
            tool_info = {
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.input_schema
            }
            tool_list.append(tool_info)

tool_list

[{'name': 'read_wiki_structure',
  'description': 'Get a list of documentation topics for a GitHub repository.\n\nArgs:\n    repoName: GitHub repository in owner/repo format (e.g. "facebook/react")',
  'input_schema': {'properties': {'repoName': {'type': 'string'}},
   'required': ['repoName'],
   'type': 'object'}},
 {'name': 'read_wiki_contents',
  'description': 'View documentation about a GitHub repository.\n\nArgs:\n    repoName: GitHub repository in owner/repo format (e.g. "facebook/react")',
  'input_schema': {'properties': {'repoName': {'type': 'string'}},
   'required': ['repoName'],
   'type': 'object'}},
 {'name': 'ask_question',
  'description': 'Ask any question about a GitHub repository and get an AI-powered, context-grounded response.\n\nArgs:\n    repoName: GitHub repository or list of repositories (max 10) in owner/repo format\n    question: The question to ask about the repository',
  'input_schema': {'properties': {'repoName': {'anyOf': [{'type': 'string'},
      {'i

---------------